# Fáza 3: Downstream Face Recognition — Full Pipeline

This notebook runs the **entire Phase 3 downstream experiment** on Google Colab with GPU:

1. **Setup** — Clone repo, install dependencies, authenticate W&B
2. **Data** — Download WebFace4M, prepare datasets
3. **Style Transfer** — Train CUT model & generate newspaper-style augmented images
4. **Training** — Train E1 (baseline), E2 (augmented), E3 (mixed)
5. **Evaluation** — Rank-1, FAR/FRR, TAR@FAR on held-out test set
6. **Comparison** — Side-by-side results table + W&B dashboard

---
## 0. Configuration

Edit these values before running:

In [ ]:
# ============================================================
#  CONFIGURATION — EDIT THESE VALUES
# ============================================================

# W&B settings
WANDB_ENTITY = "knn-proj"                   
WANDB_PROJECT = "downstream-face-rec"         

# Git
REPO_URL = "https://github.com/jetoadka/knn-proj.git"
BRANCH = "feature/downstream"

# Data — how many WebFace4M shards to use (each has ~53k images)
# 2 shards (~106k images) is good for a quick test
# 10 shards (~530k images) is recommended for real experiments
NUM_WEBFACE_SHARDS = 2

# CUT Style Transfer
CUT_EXPERIMENT_NAME = "newspaper_style_colab"  # CUT run name
CUT_N_EPOCHS = 100         # Epochs with constant LR (reduce to 20 for quick test)
CUT_N_EPOCHS_DECAY = 100   # Epochs with decaying LR (reduce to 20 for quick test)
CUT_BATCH_SIZE = 4         # Increase to 8 on A100

# Downstream Training
BACKBONE = "convnext_atto"       # lightweight backbone, fast training
LOSS = "cosface"                  # cosface | arcface | adaface
EMBEDDING_DIM = 512
BATCH_SIZE = 64                   # increase to 128-256 on A100
EPOCHS = 30                       # reduce to 5-10 for quick test
LR = 1e-3
WEIGHT_DECAY = 1e-1
EVAL_INTERVAL = 5                 # evaluate every N epochs

# Google Drive (recommended for persistent storage)
USE_GOOGLE_DRIVE = True
DRIVE_BASE = "/content/drive/MyDrive/knn-proj"

print("Configuration loaded")
print(f"   CUT training: {CUT_N_EPOCHS}+{CUT_N_EPOCHS_DECAY} epochs")
print(f"   Downstream: {EPOCHS} epochs, {BACKBONE}, {LOSS}")

---
## 1. Setup Environment

In [ ]:
# Check GPU availability
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print(" No GPU detected! Go to Runtime → Change runtime type → GPU")
    raise RuntimeError("GPU required")

In [ ]:
# Mount Google Drive
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f" Google Drive mounted: {DRIVE_BASE}")
else:
    print(" Google Drive not mounted — data will be lost on disconnect")

In [ ]:
# Clone repository
import os

WORK_DIR = "/content/knn-proj"

if not os.path.exists(WORK_DIR):
    !git clone -b {BRANCH} {REPO_URL} {WORK_DIR}
    print(f" Cloned {BRANCH}")
else:
    !cd {WORK_DIR} && git pull
    print(f" Pulled latest changes")

os.chdir(WORK_DIR)
print(f" Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q pytest scikit-learn
print("\n Dependencies installed")

In [ ]:
# Authenticate W&B
import wandb
wandb.login()
print(f" Logged in to W&B")
print(f"   Dashboard: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}")

In [ ]:
# Run unit tests to verify everything works
!python -m pytest tests/ -v --tb=short 2>&1 | tail -10

---
## 2. Download & Prepare Data

In [ ]:
import os
from pathlib import Path

# Set up directories
# We use local Colab storage for fast I/O during extraction and training
DATA_DIR = Path("/content/knn-proj/data")
CKPT_DIR = Path("/content/knn-proj/checkpoints")

if USE_GOOGLE_DRIVE:
    print(f" Using Google Drive for final backup: {DRIVE_BASE}")

DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Symlink so scripts find data at standard paths
for link_path, target in [(Path("data"), DATA_DIR), (Path("checkpoints"), CKPT_DIR)]:
    if not link_path.exists():
        os.symlink(str(target), str(link_path))

print(f" Data: {DATA_DIR}")
print(f" Checkpoints: {CKPT_DIR}")


In [ ]:
# Download WebFace4M shards from HuggingFace
WEBFACE_DIR = DATA_DIR / "webface4m"

existing_shards = list(WEBFACE_DIR.glob("*.tar.gz")) if WEBFACE_DIR.exists() else []
if len(existing_shards) >= NUM_WEBFACE_SHARDS:
    print(f" WebFace4M already downloaded ({len(existing_shards)} shards)")
else:
    print(f" Downloading {NUM_WEBFACE_SHARDS} WebFace4M shards (~{NUM_WEBFACE_SHARDS * 80} MB)...")
    !python -m src.data_prep.download_webface4m \
        --output-dir {WEBFACE_DIR} \
        --num-shards {NUM_WEBFACE_SHARDS} \
        --verify
    print(" Download complete")

In [ ]:
# Extract WebFace4M shards to flat image directory
import tarfile
from tqdm.auto import tqdm

WEBFACE_FLAT = DATA_DIR / "webface4m_flat"

if WEBFACE_FLAT.exists() and len(list(WEBFACE_FLAT.glob("*.jpg"))) > 100:
    n_existing = len(list(WEBFACE_FLAT.glob("*.jpg")))
    print(f" WebFace4M already extracted ({n_existing:,} images)")
else:
    WEBFACE_FLAT.mkdir(parents=True, exist_ok=True)
    shard_files = sorted(WEBFACE_DIR.glob("*.tar.gz"))
    print(f" Extracting {len(shard_files)} shards...")

    total_images = 0
    for shard_path in shard_files:
        print(f"  {shard_path.name}...")
        with tarfile.open(shard_path, 'r:gz') as tar:
            members = tar.getmembers()
            jpgs = {m.name.replace('.jpg', ''): m for m in members if m.name.endswith('.jpg')}
            clss = {m.name.replace('.cls', ''): m for m in members if m.name.endswith('.cls')}

            for key in tqdm(jpgs, desc=shard_path.stem, leave=False):
                if key in clss:
                    cls_f = tar.extractfile(clss[key])
                    if cls_f:
                        cls_id = cls_f.read().decode().strip()
                    else:
                        continue
                    key_basename = key.split('/')[-1] if '/' in key else key
                    out_name = f"{cls_id}_{key_basename}.jpg"
                    out_path = WEBFACE_FLAT / out_name
                    if not out_path.exists():
                        jpg_f = tar.extractfile(jpgs[key])
                        if jpg_f:
                            out_path.write_bytes(jpg_f.read())
                            total_images += 1

    print(f"\n Extracted {total_images:,} images")

---
## 3. Style Transfer (CUT) — Train & Generate

This section:
1. Checks for an **existing CUT checkpoint** (from Google Drive or a teammate)
2. If not found, **trains CUT from scratch** (~1-2h on T4)
3. Uses the generator to **augment WebFace4M** with newspaper-style images

> 💡 If you already have a trained checkpoint, upload `latest_net_G.pth` to
> `MyDrive/knn-proj/cut_checkpoints/` and skip training.

In [ ]:
# Download missing CUT packages from the original repo
import os
CUT_DIR = "src/style_transfer/cut_model"

if not os.path.exists(f"{CUT_DIR}/models"):
    !git clone --depth 1 https://github.com/taesungp/contrastive-unpaired-translation.git /tmp/cut_orig
    !cp -r /tmp/cut_orig/models {CUT_DIR}/models
    !cp -r /tmp/cut_orig/data {CUT_DIR}/data
    !rm -rf /tmp/cut_orig
    print("CUT models/ and data/ packages installed")
else:
    print(" CUT packages already present")


In [ ]:
from pathlib import Path
import os

# Check for existing CUT checkpoint
CUT_CKPT_DIR = Path(f"src/style_transfer/cut_model/checkpoints/{CUT_EXPERIMENT_NAME}")
DRIVE_CUT_CKPT = Path(DRIVE_BASE) / "cut_checkpoints" if USE_GOOGLE_DRIVE else None

cut_checkpoint_found = False

# 1. Attempt to download from shared Google Drive first
if not cut_checkpoint_found:
    print("\u23f3 Attempting to download CUT model from shared Google Drive...")
    !pip install -q -U gdown
    !gdown --folder https://drive.google.com/drive/folders/1EOveqcsUhdTKPMqbYeaLpQGrccafj5sp -O /tmp/shared_cut
    
    # It might create a nested folder depending on the Drive structure
    shared_model_path = Path("/tmp/shared_cut/CUT_models/latest_net_G.pth")
    # Fallback to recursively finding it just in case structure differs slightly
    if not shared_model_path.exists():
        found = list(Path("/tmp/shared_cut").rglob("latest_net_G.pth"))
        if len(found) > 0:
            shared_model_path = found[0]
            
    if shared_model_path.exists():
        CUT_CKPT_DIR.mkdir(parents=True, exist_ok=True)
        !cp -v "{shared_model_path}" {CUT_CKPT_DIR}/
        cut_checkpoint_found = True
        print(f"\u2705 CUT checkpoint successfully downloaded from shared Drive")

# 2. Check local
if not cut_checkpoint_found and CUT_CKPT_DIR.exists() and list(CUT_CKPT_DIR.glob("*_net_G.pth")):
    cut_checkpoint_found = True
    print(f"\u2705 CUT checkpoint found locally")

# 3. Check Google Drive
elif not cut_checkpoint_found and DRIVE_CUT_CKPT and DRIVE_CUT_CKPT.exists() and list(DRIVE_CUT_CKPT.glob("*_net_G.pth")):
    CUT_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    !cp -v {DRIVE_CUT_CKPT}/*net*.pth {CUT_CKPT_DIR}/
    cut_checkpoint_found = True
    print(f"\u2705 CUT checkpoint restored from Google Drive")

# 4. Check legacy path (teammate's checkpoint)
elif not cut_checkpoint_found:
    legacy_dir = Path("src/style_transfer/cut_model/checkpoints/exp_combined")
    if legacy_dir.exists() and list(legacy_dir.glob("*_net_G.pth")):
        CUT_CKPT_DIR = legacy_dir
        cut_checkpoint_found = True
        print(f"\u2705 CUT checkpoint found at legacy path")

if not cut_checkpoint_found:
    print("\u26a0\ufe0f No CUT checkpoint found \u2014 will train from scratch")
    print(f"   Training: {CUT_N_EPOCHS}+{CUT_N_EPOCHS_DECAY} epochs")


In [ ]:
# 3.1 Prepare CUT training data (trainA=clean, trainB=noisy)
CUT_DATA_DIR = Path("src/style_transfer/cut_dataset_colab")

if cut_checkpoint_found:
    print(" CUT already trained, skipping data prep")
elif (CUT_DATA_DIR / "trainA").exists() and (CUT_DATA_DIR / "trainB").exists():
    trainA_count = len(list((CUT_DATA_DIR / "trainA").glob("*")))
    trainB_count = len(list((CUT_DATA_DIR / "trainB").glob("*")))
    print(f" CUT data ready: trainA={trainA_count}, trainB={trainB_count}")
else:
    print(" Preparing CUT training data...")
    print(f"   trainA (clean): WebFace4M")
    print(f"   trainB (noisy): people_gator + wiki_face_112")

    !python src/style_transfer/prepare_cut_data.py \
        --clean {WEBFACE_FLAT} \
        --noisy sample_data/people_gator/aligned_112/train sample_data/wiki_face_112 \
        --output {CUT_DATA_DIR}

    print(" CUT data prepared")

In [ ]:
# 3.1b Install missing CUT packages (models/ and data/)
# The original CUT repo (taesungp/contrastive-unpaired-translation) includes
# 'models' and 'data' Python packages that were not committed to our repo.
import os

CUT_DIR = 'src/style_transfer/cut_model'
if not os.path.exists(f'{CUT_DIR}/models'):
    print('⬇️ Downloading missing CUT packages (models/, data/)...')
    !git clone --depth 1 -q https://github.com/taesungp/contrastive-unpaired-translation.git /tmp/cut_orig
    !cp -r /tmp/cut_orig/models {CUT_DIR}/models
    !cp -r /tmp/cut_orig/data {CUT_DIR}/data
    !rm -rf /tmp/cut_orig
    print('✅ CUT models/ and data/ packages installed')
else:
    print('✅ CUT packages already present')

In [ ]:
# 3.2 Train CUT model
import os

if cut_checkpoint_found:
    print(" CUT checkpoint exists, skipping training")
else:
    print(f" Training CUT model...")
    print(f"   Experiment: {CUT_EXPERIMENT_NAME}")
    print(f"   Epochs: {CUT_N_EPOCHS} + {CUT_N_EPOCHS_DECAY} (decay)")
    print(f"   W&B project: style-transfer")
    print(f"   Expected time: ~1-2 hours on T4\n")

    os.environ['WANDB_ENTITY'] = WANDB_ENTITY

    !cd src/style_transfer/cut_model && python train.py \
        --dataroot ../cut_dataset_colab \
        --name {CUT_EXPERIMENT_NAME} \
        --model cut \
        --load_size 112 \
        --crop_size 112 \
        --batch_size {CUT_BATCH_SIZE} \
        --n_epochs {CUT_N_EPOCHS} \
        --n_epochs_decay {CUT_N_EPOCHS_DECAY} \
        --display_freq 200 \
        --gpu_ids 0 \
        --display_id -1

    # Verify
    if CUT_CKPT_DIR.exists() and list(CUT_CKPT_DIR.glob("*_net_G.pth")):
        cut_checkpoint_found = True
        print(f"\n CUT training complete!")
    else:
        print(f"\n CUT training failed — check output above")

In [ ]:
# 3.3 Save CUT checkpoint to Google Drive
if cut_checkpoint_found and USE_GOOGLE_DRIVE and DRIVE_CUT_CKPT:
    DRIVE_CUT_CKPT.mkdir(parents=True, exist_ok=True)
    !cp -v {CUT_CKPT_DIR}/*net*.pth {DRIVE_CUT_CKPT}/
    print(f" CUT checkpoint backed up to: {DRIVE_CUT_CKPT}")

In [ ]:
# 3.4 Generate newspaper-style augmented data
AUGMENTED_DIR = DATA_DIR / "augmented_newspaper"

if not cut_checkpoint_found:
    print(" No CUT checkpoint — only E1 (baseline) will be available")
elif AUGMENTED_DIR.exists() and len(list(AUGMENTED_DIR.rglob("*.jpg"))) > 100:
    n_aug = len(list(AUGMENTED_DIR.rglob("*.jpg")))
    print(f" Augmented data exists ({n_aug:,} images)")
else:
    print(f" Generating newspaper-style images...")
    print(f"   Input:  {WEBFACE_FLAT}")
    print(f"   Output: {AUGMENTED_DIR}")
    print(f"   ~30-60 minutes on T4...\n")

    !python -m src.downstream.generate_augmented_data \
        --input-dir {WEBFACE_FLAT} \
        --output-dir {AUGMENTED_DIR} \
        --checkpoint-dir {CUT_CKPT_DIR} \
        --batch-size 32 \
        --device cuda

    n_gen = len(list(AUGMENTED_DIR.rglob("*.jpg")))
    print(f"\n Generated {n_gen:,} newspaper-style images")

---
## 4. Prepare Training Datasets

- **E1**: WebFace4M only (baseline)
- **E2**: WebFace4M + CUT-generated newspaper-style
- **E3**: WebFace4M + CUT-generated + real newspaper (people_gator train)

In [ ]:
DOWNSTREAM_DIR = DATA_DIR / "downstream"

# E1: Baseline (clean only)
E1_DIR = DOWNSTREAM_DIR / "E1_baseline" / "train"
if E1_DIR.exists() and any(E1_DIR.iterdir()):
    print(" E1 dataset ready")
else:
    print(" Preparing E1 (baseline)...")
    !python -m src.downstream.prepare_timm_dataset \
        --sources {WEBFACE_FLAT} \
        --output-dir {E1_DIR} --mode train --symlink

# E2: Augmented (clean + CUT-generated)
if cut_checkpoint_found:
    E2_DIR = DOWNSTREAM_DIR / "E2_augmented" / "train"
    if E2_DIR.exists() and any(E2_DIR.iterdir()):
        print(" E2 dataset ready")
    else:
        print(" Preparing E2 (augmented)...")
        !python -m src.downstream.prepare_timm_dataset \
            --sources {WEBFACE_FLAT} {AUGMENTED_DIR} \
            --output-dir {E2_DIR} --mode train --symlink

    # E3: Mixed
    E3_DIR = DOWNSTREAM_DIR / "E3_mixed" / "train"
    GATOR_TRAIN = Path("sample_data/people_gator/aligned_112/train")
    if E3_DIR.exists() and any(E3_DIR.iterdir()):
        print(" E3 dataset ready")
    else:
        print(" Preparing E3 (mixed)...")
        !python -m src.downstream.prepare_timm_dataset \
            --sources {WEBFACE_FLAT} {AUGMENTED_DIR} {GATOR_TRAIN} \
            --output-dir {E3_DIR} --mode train --symlink
else:
    print(" Skipping E2/E3 (no CUT data)")

print("\n Dataset preparation complete")

---
## 5. Training

Train each experiment with live W&B logging.

> 💡 Monitor in real-time at your [W&B dashboard](https://wandb.ai)!

In [ ]:
import subprocess, sys, os

os.environ['WANDB_ENTITY'] = WANDB_ENTITY
os.environ['WANDB_PROJECT'] = WANDB_PROJECT

def train_experiment(run_name, train_dir, val_dir="sample_data/people_gator/aligned_112/dev"):
    """Train a face recognition experiment."""
    cmd = [
        sys.executable, "-m", "src.downstream.train_downstream",
        "--train-dir", str(train_dir),
        "--val-dir", str(val_dir),
        "--backbone", BACKBONE, "--loss", LOSS,
        "--embedding-dim", str(EMBEDDING_DIM),
        "--batch-size", str(BATCH_SIZE),
        "--epochs", str(EPOCHS),
        "--lr", str(LR), "--weight-decay", str(WEIGHT_DECAY),
        "--eval-interval", str(EVAL_INTERVAL),
        "--run-name", run_name,
        "--wandb-mode", "online",
        "--save-dir", str(CKPT_DIR),
        "--device", "cuda", "--num-workers", "2",
    ]
    print(f"\n{'='*60}")
    print(f"🚀 {run_name}")
    print(f"   Train: {train_dir}")
    print(f"   W&B:   https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}")
    print(f"{'='*60}\n")
    result = subprocess.run(cmd, env={**os.environ})
    print(f"\n{run_name} {'complete' if result.returncode == 0 else 'FAILED'}")
    return result.returncode

print(" Training helper ready")

In [ ]:
# E1: Baseline (clean only)
train_experiment("E1-baseline-clean", DOWNSTREAM_DIR / "E1_baseline" / "train")

In [ ]:
# E2: Augmented (clean + CUT-generated)
if cut_checkpoint_found:
    train_experiment("E2-augmented-newspaper", DOWNSTREAM_DIR / "E2_augmented" / "train")
else:
    print(" Skipping E2")

In [ ]:
# E3: Mixed (clean + CUT-generated + real newspaper)
if cut_checkpoint_found:
    train_experiment("E3-augmented-mixed", DOWNSTREAM_DIR / "E3_mixed" / "train")
else:
    print(" Skipping E3")

---
## 6. Evaluation

Evaluate all trained models on the held-out people_gator test set.

In [ ]:
# Find trained checkpoints
EVAL_TEST_DIR = Path("sample_data/people_gator/aligned_112/test")
Path("results").mkdir(exist_ok=True)

experiments = [("E1-baseline-clean", CKPT_DIR / "E1-baseline-clean" / "best_model.pth")]
if cut_checkpoint_found:
    experiments += [
        ("E2-augmented-newspaper", CKPT_DIR / "E2-augmented-newspaper" / "best_model.pth"),
        ("E3-augmented-mixed", CKPT_DIR / "E3-augmented-mixed" / "best_model.pth"),
    ]

# Filter to existing checkpoints (try epoch_N.pth as fallback)
valid = []
for name, path in experiments:
    if path.exists():
        valid.append((name, path))
    else:
        fallbacks = sorted(path.parent.glob("epoch_*.pth")) if path.parent.exists() else []
        if fallbacks:
            valid.append((name, fallbacks[-1]))
            print(f"  Using fallback: {fallbacks[-1].name}")

print(f"\n {len(valid)} experiments to evaluate:")
for n, p in valid:
    print(f"   {n}: {p.name}")

In [ ]:
# Run evaluation
if valid:
    ckpt_str = " ".join(str(p) for _, p in valid)
    name_str = " ".join(n for n, _ in valid)

    !python -m src.downstream.evaluate \
        --checkpoint {ckpt_str} \
        --test-dir {EVAL_TEST_DIR} \
        --backbone {BACKBONE} \
        --embedding-dim {EMBEDDING_DIM} \
        --experiment-name {name_str} \
        --device cuda \
        --wandb-mode online \
        --output results/final_comparison.json
else:
    print(" No checkpoints to evaluate")

In [ ]:
# Display results table
import json

rf = Path("results/final_comparison.json")
if rf.exists():
    results = json.load(open(rf))
    print(f"\n{'='*70}")
    print(f"🏆 FINAL RESULTS")
    print(f"{'='*70}")
    print(f"{'Experiment':<28} {'Rank-1':>8} {'Rank-5':>8} {'TAR@1e-4':>10} {'10-fold':>8}")
    print('-'*70)
    for r in results:
        n = r.get('experiment','?')
        r1 = f"{r['rank1_accuracy']:.4f}" if r.get('rank1_accuracy') is not None else '—'
        r5 = f"{r['rank5_accuracy']:.4f}" if r.get('rank5_accuracy') is not None else '—'
        t  = f"{r['tar_at_far_1e4']:.4f}" if r.get('tar_at_far_1e4') is not None else '—'
        k  = f"{r['kfold_verification_accuracy']:.4f}" if r.get('kfold_verification_accuracy') is not None else '—'
        print(f"{n:<28} {r1:>8} {r5:>8} {t:>10} {k:>8}")
    print('-'*70)
else:
    print("No results yet")

---
## 7. W&B Comparison

In [ ]:
!python -m src.evaluation.compare_experiments \
    --entity {WANDB_ENTITY} --project {WANDB_PROJECT} \
    --markdown results/comparison_report.md \
    --output results/comparison_data.json

print(f"\n Dashboard: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}")

In [ ]:
from IPython.display import Markdown, display
rp = Path("results/comparison_report.md")
if rp.exists():
    display(Markdown(rp.read_text()))
else:
    print("No report yet")

---
## 8. Save to Google Drive

In [ ]:
if USE_GOOGLE_DRIVE:
    import shutil
    drive_results = Path(DRIVE_BASE) / "results"
    drive_results.mkdir(parents=True, exist_ok=True)
    for f in Path("results").glob("*"):
        shutil.copy2(f, drive_results / f.name)
        print(f" {f.name}")
    print(f"\nResults saved to {drive_results}")

    # Export (save) downstream models to Google Drive
    drive_checkpoints = Path(DRIVE_BASE) / "checkpoints"
    drive_checkpoints.mkdir(parents=True, exist_ok=True)
    print("\nBacking up checkpoints to Google Drive...")
    !cp -r {CKPT_DIR}/* {drive_checkpoints}/
    print(f" Models exported to {drive_checkpoints}")
else:
    print(" Download results before disconnecting!")
    from google.colab import files
    if Path("results/final_comparison.json").exists():
        files.download("results/final_comparison.json")
